# Unit 2b — trap time, entries, first-hit weights and committor (logit, graph-train only, Nov 17, 2026)

Same chain as Unit 2a (`src/slider/markov.py`), start µ = uniform over all 128 states; traps and plateau_id from
`u1_traps.csv`. Per graph × T: Yc, Yb (steps in traps), entries, duration = Yb / entries, P_hit, Ya (first-hit
weighted tau). Per plateau × T: committor qbar (consensus before returning to the plateau, averaged over the exit
flux), split into downhill and uphill exits; attempts = 1 / qbar. Graph-test graphs raise
(`markov.graph_trap_time`).

In [1]:
import pandas as pd

from slider import markov

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
G, Pl, secs = markov.save_tables_2b()
s = markov.summary_2b(G, Pl)
print(f"{G['atlas_id'].nunique()} graphs, {len(G)} graph rows, {len(Pl)} plateau rows; {secs:.1f} s")

597 graphs, 7761 graph rows, 12649 plateau rows; 17.2 s


## Human networks: Yb, entries, duration, Ya, P_hit by T

In [2]:
s["humans"].unstack("human_name").round(4)

Yb                               entries                               duration                                         Ya                                  P_hit                          
human_name           CT   Hierarchy     Path     Ring      CT Hierarchy    Path    Ring           CT   Hierarchy     Path     Ring           CT   Hierarchy      Path     Ring      CT Hierarchy    Path    Ring
T                                                                                                                                                                                                               
0.10        177495.8956  32898.2221  32.3909  24.6162  1.1513    1.0668  1.3194  1.6089  154164.2628  30838.4521  24.5505  15.2996  385433.1444  85225.1876   51.9205  37.5992  0.4605    0.3861  0.7423  0.8045
0.15          6313.1991   1173.8753  32.2766  24.6161  1.1522    1.0657  1.3293  1.6090    5479.4662   1101.4804  24.2817  15.2994   13721.1724   3057.3055   51.8437  37.5992  0.4607    0.3864  0.7423  0.8045
0.20          1176.8640    222.0170  31.7763  24.6156  1.1558    1.0614  1.3726  1.6094    1018.2191    209.1765  23.1502  15.2948    2568.1395    592.0212   51.5101  37.6011  0.4614    0.3878  0.7423  0.8045
0.30           207.9729     42.5838  29.3515  24.5998  1.1758    1.0480  1.5846  1.6219     176.8809     40.6336  18.5234  15.1672     465.2107    127.8437   49.9668  37.6535  0.4653    0.3955  0.7426  0.8048
0.40            82.0328     19.3630  26.1430  24.5326  1.2092    1.0507  1.8759  1.6772      67.8411     18.4286  13.9361  14.6274     193.2642     68.9705   48.2033  37.8911  0.4718    0.4092  0.7440  0.8060
0.50            45.1273     12.5894  23.2843  24.4066  1.2482    1.0764  2.1609  1.7921      36.1532     11.6962  10.7752  13.6192     114.9272     53.3585   47.1242  38.4162  0.4795    0.4264  0.7470  0.8087
0.60            29.7073      9.7553  21.0890  24.2550  1.2874    1.1163  2.4160  1.9595      23.0762      8.7387   8.7291  12.3784      83.3853     48.4267   46.9287  39.2553  0.4876    0.4443  0.7513  0.8125
0.80            17.3118      7.4403  18.3322  24.0250  1.3580    1.2080  2.8440  2.3840      12.7479      6.1594   6.4460  10.0776      60.5955     47.9853   48.5867  41.7700  0.5031    0.4760  0.7618  0.8221
1.00            12.5079      6.5011  16.8801  23.9889  1.4173    1.2929  3.1913  2.8411       8.8252      5.0283   5.2895   8.4435      54.4727     51.3630   51.7932  45.0752  0.5170    0.5005  0.7725  0.8320
1.50             8.2773      5.5954  15.4717  24.4932  1.5288    1.4511  3.8299  3.8647       5.4143      3.8559   4.0398   6.3376      56.0336     62.4069   61.6284  54.5856  0.5436    0.5390  0.7935  0.8521
2.00             6.8733      5.2538  15.0987  25.2550  1.6061    1.5537  4.2617  4.6518       4.2794      3.3814   3.5429   5.4291      62.7989     72.4694   70.9320  63.7692  0.5612    0.5605  0.8071  0.8651
3.00             5.8407      4.9785  15.0198  26.6017  1.7049    1.6757  4.8021  5.7152       3.4258      2.9710   3.1277   4.6546      76.3850     87.6782   85.4752  78.6657  0.5821    0.5833  0.8225  0.8798
5.00             5.2418      4.8106  15.2146  28.2701  1.8045    1.7904  5.3377  6.8362       2.9049      2.6870   2.8504   4.1354      95.3591    105.6459  103.0721  97.4564  0.6013    0.6027  0.8358  0.8923

## T* where Yb(Hierarchy) = Yb(Ring) (linear interpolation of log Yb between grid points)

In [3]:
s["T_star"]

[0.36986950826181997]

## qbar over graph-train plateaus, by B and T

In [4]:
s["qbar_by_B"].round(4)

n_plateaus  median     q25     q75     IQR
B T                                               
0 0.10         899  0.9999  0.9999  1.0000  0.0001
  0.15         899  0.9984  0.9976  0.9992  0.0016
  0.20         899  0.9918  0.9872  0.9956  0.0084
  0.30         899  0.9578  0.9370  0.9755  0.0385
  0.40         899  0.9068  0.8693  0.9387  0.0695
  0.50         899  0.8542  0.8030  0.8953  0.0922
  0.60         899  0.8029  0.7512  0.8457  0.0945
  0.80         899  0.7161  0.6666  0.7608  0.0942
  1.00         899  0.6582  0.6103  0.7035  0.0932
  1.50         899  0.5721  0.5130  0.6241  0.1111
  2.00         899  0.5270  0.4564  0.5694  0.1130
  3.00         899  0.4726  0.4012  0.5355  0.1343
  5.00         899  0.4310  0.3422  0.5055  0.1633
1 0.10          74  0.5198  0.4167  0.5754  0.1587
  0.15          74  0.5196  0.4166  0.5755  0.1589
  0.20          74  0.5185  0.4163  0.5760  0.1597
  0.30          74  0.5217  0.4154  0.5762  0.1608
  0.40          74  0.5101  0.4162  0.5732  0.1570
  0.50          74  0.5033  0.4314  0.5713  0.1399
  0.60          74  0.5017  0.4434  0.5711  0.1278
  0.80          74  0.5128  0.4641  0.5650  0.1009
  1.00          74  0.5265  0.4837  0.5627  0.0790
  1.50          74  0.5380  0.5077  0.5703  0.0626
  2.00          74  0.5438  0.5219  0.5776  0.0557
  3.00          74  0.5554  0.5272  0.5819  0.0547
  5.00          74  0.5582  0.5417  0.5762  0.0345